In [7]:
# Cell 1 — Imports & Config
import os, json, pickle, re, math, warnings
import numpy as np
import pandas as pd
from pathlib import Path

In [8]:
# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [11]:
# Paths (edit DATA_FOLDER to your dataset location)
DATA_FOLDER = Path('D:/Projects/Amazon ML Challenge/68e8d1d70b66d_student_resource/student_resource/faster_image_pricing/faster_image_pricing_mm/dataset')   # expects train.csv and test.csv
ARTIFACTS = Path('./artifacts')
ARTIFACTS.mkdir(parents=True, exist_ok=True)

In [12]:
# Cell 2 — Load data
train_path = DATA_FOLDER / 'train.csv'
test_path  = DATA_FOLDER / 'test.csv'

train = pd.read_csv(train_path)
test  = pd.read_csv(test_path)

# Normalize column names to lowercase for robustness
train.columns = [c.lower() for c in train.columns]
test.columns  = [c.lower() for c in test.columns]

print("train:", train.shape, "test:", test.shape)
display(train.head(2))


train: (75000, 4) test: (75000, 3)


,sample_id,catalog_content,image_link,price
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.89
1,198967,"Item Name: Salerno Cookies, The Original Butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.12


In [13]:
# Cell 3 — Utilities: SMAPE and lightweight extractors
def smape(y_true, y_pred):
    # Symmetric Mean Absolute Percentage Error (in %), robust to zeros.
    # Expects raw (non-log) targets and predictions.
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    denom[denom == 0] = 1.0
    return np.mean(np.abs(y_pred - y_true) / denom) * 100.0

# If value/unit/pack are missing, try to extract them from a 'catalog_content' or similar text column.
def try_extract_value_unit_pack(df, text_col='catalog_content'):
    if text_col not in df.columns:
        return df  # nothing to do
    s = df[text_col].astype(str).fillna('')

    # Value: capture a float or int after 'value' or standalone number before unit
    val = s.str.extract(r'(?:value\s*[:=]\s*)?(\d+(?:[\.,]\d+)?)\s*(?:[a-zA-Z]+)?', expand=False)
    # Unit: capture common units (oz, ml, g, kg, lb, l)
    unit = s.str.extract(r'\b(oz|ml|g|kg|lb|l|litre|liter)\b', expand=False)
    # Pack: capture 'pack: N' or 'xN' patterns
    pack = s.str.extract(r'(?:pack\s*[:=]\s*)?(\d+)\s*(?:pack|pcs|x)?|x\s*(\d+)', expand=True)
    pack = pack[0].fillna(pack[1])

    # Only fill if columns are missing
    if 'value' not in df.columns:
        df['value'] = pd.to_numeric(val.str.replace(',', '.', regex=False), errors='coerce')
    if 'unit' not in df.columns:
        df['unit'] = unit
    if 'pack' not in df.columns:
        df['pack'] = pd.to_numeric(pack, errors='coerce')

    return df


In [14]:
# Cell 4 — Basic cleaning & target engineering
# Ensure essential numeric columns exist; if not, attempt light extraction
for frame in (train, test):
    try_extract_value_unit_pack(frame)

# Impute selected numeric columns
num_cols_base = ['value', 'value_base_oz', 'total_qty_oz']
for col in num_cols_base:
    if col in train.columns:
        med = train[col].median(skipna=True)
        train[col] = train[col].fillna(med)
        if col in test.columns:
            test[col] = test[col].fillna(med)
    else:
        # if missing, create zeros
        train[col] = 0.0
        test[col]  = 0.0

# Pack as numeric
if 'pack' in train.columns:
    train['pack'] = pd.to_numeric(train['pack'], errors='coerce').fillna(1).astype(int)
else:
    train['pack'] = 1
if 'pack' in test.columns:
    test['pack'] = pd.to_numeric(test['pack'], errors='coerce').fillna(1).astype(int)
else:
    test['pack'] = 1

# Derive 'price_per_totalqty' if price / total_qty_oz available (train only)
price_col = 'price' if 'price' in train.columns else ('price' if 'Price' not in train.columns else 'Price')
if 'price' not in train.columns and 'Price' in train.columns:
    train = train.rename(columns={'Price': 'price'})

train['price_per_totalqty'] = train['price'] / train['total_qty_oz'].replace(0, np.nan)
train['price_per_totalqty'] = train['price_per_totalqty'].fillna(train['price'])

# Log target
train['log_price'] = np.log1p(train['price'])

print("price stats:"); display(train['price'].describe())
print("log_price stats:"); display(train['log_price'].describe())


price stats:


count    75000.000000
mean        23.647654
std         33.376932
min          0.130000
25%          6.795000
50%         14.000000
75%         28.625000
max       2796.000000
Name: price, dtype: float64

log_price stats:


count    75000.000000
mean         2.739217
std          0.942032
min          0.122218
25%          2.053483
50%          2.708050
75%          3.388619
max          7.936303
Name: log_price, dtype: float64

In [15]:
# Cell 5 — Encode 'unit' categorically
if 'unit' not in train.columns:
    train['unit'] = ''
if 'unit' not in test.columns:
    test['unit'] = ''

units = pd.Series(list(train['unit'].astype(str)) + list(test['unit'].astype(str))).unique().tolist()
unit_mapping = {u: i for i, u in enumerate(sorted(units))}
train['unit_enc'] = train['unit'].map(unit_mapping).fillna(-1).astype(int)
test['unit_enc']  = test['unit'].map(unit_mapping).fillna(-1).astype(int)

print("Unique units mapped:", len(unit_mapping))
display(train[['unit','unit_enc']].head())


Unique units mapped: 9


,unit,unit_enc
0,NaN,-1
1,NaN,-1
2,NaN,-1
3,oz,8
4,NaN,-1


In [16]:
# Cell 6 — Text features (TF-IDF)
from sklearn.feature_extraction.text import TfidfVectorizer

# Choose a text column; default to 'catalog_content' else try a likely alternative
text_col_candidates = [c for c in ['catalog_content','content','description','name','title'] if c in train.columns]
assert len(text_col_candidates) > 0, "No suitable text column found (expected one of catalog_content/content/description/name/title)."
TEXT_COL = text_col_candidates[0]

tfidf = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1,2))
X_text_train = tfidf.fit_transform(train[TEXT_COL].fillna(''))
X_text_test  = tfidf.transform(test[TEXT_COL].fillna(''))
print("TF-IDF shapes:", X_text_train.shape, X_text_test.shape)


TF-IDF shapes: (75000, 5000) (75000, 5000)


In [17]:
# Cell 7 — Combine numeric + text features
from scipy.sparse import hstack, csr_matrix

num_cols = ['value', 'value_base_oz', 'total_qty_oz', 'pack', 'unit_enc']
for col in num_cols:
    if col not in train.columns:
        train[col] = 0.0
        test[col] = 0.0

X_num_train = train[num_cols].astype(float).values
X_num_test  = test[num_cols].astype(float).values

X_train = hstack([X_text_train, csr_matrix(X_num_train)])
X_test  = hstack([X_text_test,  csr_matrix(X_num_test)])

y = train['log_price'].values
print("Combined shapes:", X_train.shape, X_test.shape)


Combined shapes: (75000, 5005) (75000, 5005)


In [18]:
# Cell 8 — Train/Val split and LightGBM training
import lightgbm as lgb
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(X_train, y, test_size=0.2, random_state=RANDOM_STATE)

lgb_train = lgb.Dataset(X_tr, label=y_tr)
lgb_val   = lgb.Dataset(X_val, label=y_val)

params = {
    'objective': 'regression',
    'metric': 'l2',             # we'll compute SMAPE on back-transformed values
    'learning_rate': 0.05,
    'num_leaves': 64,
    'verbosity': -1,
    'seed': RANDOM_STATE,
    'n_jobs': -1
}

print("Training LightGBM...")
bst = lgb.train(
    params,
    lgb_train,
    num_boost_round=2000,
    valid_sets=[lgb_train, lgb_val],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=100)]
)


Training LightGBM...
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1061]	training's l2: 0.205793	valid_1's l2: 0.480562


In [19]:
# Cell 9 — Evaluate validation SMAPE (back-transform from log)
y_val_pred_log = bst.predict(X_val, num_iteration=bst.best_iteration)
y_val_pred = np.expm1(y_val_pred_log)
y_val_true = np.expm1(y_val)

val_smape = smape(y_val_true, y_val_pred)
print(f"Validation SMAPE: {val_smape:.6f}%")


Validation SMAPE: 52.303787%


In [20]:
# Cell 10 — Train on full data and save all artifacts
import lightgbm as lgb

full_train = lgb.Dataset(X_train, label=y)

final_bst = lgb.train(
    params,
    full_train,
    num_boost_round=int(bst.best_iteration or 1000)
)

# Save LightGBM model
model_path = ARTIFACTS / 'lgbm_model.txt'
final_bst.save_model(str(model_path))
print("Saved model to:", model_path)

# Save TF-IDF vectorizer
with open(ARTIFACTS / 'tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
print("Saved TF-IDF to:", ARTIFACTS / 'tfidf_vectorizer.pkl')

# Save unit mapping
with open(ARTIFACTS / 'unit_mapping.json', 'w') as f:
    json.dump(unit_mapping, f)
print("Saved unit_mapping to:", ARTIFACTS / 'unit_mapping.json')

# Save numeric columns list and text column name
with open(ARTIFACTS / 'config.json', 'w') as f:
    json.dump({
        'num_cols': num_cols,
        'text_col': TEXT_COL,
        'random_state': RANDOM_STATE
    }, f, indent=2)
print("Saved config to:", ARTIFACTS / 'config.json')


Saved model to: artifacts\lgbm_model.txt
Saved TF-IDF to: artifacts\tfidf_vectorizer.pkl
Saved unit_mapping to: artifacts\unit_mapping.json
Saved config to: artifacts\config.json


In [21]:
# Cell 11 — Inference helper & prediction export
import lightgbm as lgb

def load_artifacts(art_dir=ARTIFACTS):
    with open(art_dir / 'tfidf_vectorizer.pkl','rb') as f:
        vec = pickle.load(f)
    with open(art_dir / 'unit_mapping.json','r') as f:
        umap = json.load(f)
    with open(art_dir / 'config.json','r') as f:
        cfg = json.load(f)
    model = lgb.Booster(model_file=str(art_dir / 'lgbm_model.txt'))
    return vec, umap, cfg, model

def preprocess_for_inference(df, vec, umap, cfg):
    df = df.copy()
    # columns to lowercase
    df.columns = [c.lower() for c in df.columns]

    # ensure basics present
    if 'unit' not in df.columns: df['unit'] = ''
    for c in ['value','value_base_oz','total_qty_oz']:
        if c not in df.columns: df[c] = 0.0

    # unit encoding (unseen -> -1)
    df['unit_enc'] = df['unit'].map(umap).fillna(-1).astype(int)

    # numeric block
    X_num = df[cfg['num_cols']].astype(float).values

    # text
    text_col = cfg['text_col']
    X_txt = vec.transform(df[text_col].fillna(''))

    from scipy.sparse import hstack, csr_matrix
    X = hstack([X_txt, csr_matrix(X_num)])
    return X

# Predict on test.csv and write a sample submission (if id col exists)
vec, umap, cfg, model = load_artifacts()
X_test_inf = preprocess_for_inference(test, vec, umap, cfg)
pred_log = model.predict(X_test_inf)
pred = np.expm1(pred_log)

sub = pd.DataFrame({'prediction': pred})
# if test has id-like column, use it
id_col = None
for c in ['id','row_id','sample_id','test_id']:
    if c in test.columns:
        id_col = c; break
if id_col:
    sub.insert(0, id_col, test[id_col])

out_path = ARTIFACTS / 'test_out.csv'
sub.to_csv(out_path, index=False)
print("Wrote predictions to:", out_path)
display(sub.head())


Wrote predictions to: artifacts\predictions.csv


,sample_id,prediction
0,100179,14.902654
1,245611,15.907706
2,146263,17.071520
3,95658,16.224768
4,36806,34.823727
